# Análisis Exploratorio de Datos (EDA): Colombia EVA

**Propósito del trabajo:** preparar un conjunto de datos listo para entrenamiento de modelos de aprendizaje automático, respondiendo:

- ¿Qué información contienen los datos?
- ¿Qué problemas presentan?
- ¿Qué transformaciones son necesarias?
- ¿El dataset es adecuado para aprendizaje automático?

**Materia:** Programación Avanzada

**Profesor:** Jaime Alberto Vergara Tejada

**Estudiante:** Diego Alejandro Ríos Vásquez

**Fuente de datos:** Datos.gov.co (API Socrata SODA)

**Dataset ID:** 2pnw-mmge

**Endpoint:** https://www.datos.gov.co/api/v3/views/2pnw-mmge/query.json


## 1. Configuración del entorno e importaciones

In [1]:
# Instala sodapy solo si no está disponible en el entorno
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('sodapy') is None:
    print('Instalando sodapy...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'sodapy', '-q'])
    print('sodapy instalado correctamente.')
else:
    print('sodapy ya está disponible.')

sodapy ya está disponible.


In [2]:
import sys
import os
import warnings
from datetime import datetime

# Avoid circular import caused by local module shadowing pandas (e.g., pandas.py)
cwd = os.getcwd()
removed_paths = []

for p in (cwd, ''):
	if p in sys.path:
		sys.path.remove(p)
		removed_paths.append(p)

if 'pandas' in sys.modules:
	del sys.modules['pandas']

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sodapy import Socrata

# Restore removed paths
for p in reversed(removed_paths):
	sys.path.insert(0, p)

warnings.filterwarnings('ignore')

# Configure visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)

print(' Libraries imported successfully!')
print(f' Execution timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

 Libraries imported successfully!
 Execution timestamp: 2026-04-24 11:19:09


## 2. Configuración de la fuente de datos (Datos.gov.co)

In [ ]:
# Configuración para API Socrata (Datos.gov.co)
# Documentación: https://datos.gov.co/

DOMAIN = 'datos.gov.co'
DATASET_ID = '2pnw-mmge'  # Dataset oficial Colombia EVA

# Variables de entorno recomendadas:
# - DATOSABIERTOS_API_KEY
# - DATOSABIERTOS_API_SECRET
# - DATOSABIERTOS_APP_TOKEN (opcional; si no existe, se usa API_KEY como token)
API_KEY_ID = os.environ.get('DATOSABIERTOS_API_KEY')
API_SECRET = os.environ.get('DATOSABIERTOS_API_SECRET')
APP_TOKEN = os.environ.get('DATOSABIERTOS_APP_TOKEN') or API_KEY_ID

if APP_TOKEN and API_SECRET:
    auth_msg = 'Configurada (API Key + Secret)';
elif APP_TOKEN:
    auth_msg = 'Configurada (solo token/app token)'
else:
    auth_msg = 'Sin credenciales (modo publico; puede tener limites de tasa)'

print('Configuracion de Datos.gov.co:')
print(f'  Dominio: {DOMAIN}')
print(f'  Dataset ID: {DATASET_ID}')
print(f'  Dataset: Colombia EVA')
print(f'  Autenticacion: {auth_msg}')
print(f'  API Key detectada: {"Si" if API_KEY_ID else "No"}')
print(f'  API Secret detectado: {"Si" if API_SECRET else "No"}')

Configuración de Datos.gov.co:
  Dominio: datos.gov.co
  Dataset ID: 2pnw-mmge
  Dataset: Colombia EVA
  Autenticación: Sin token (modo público; puede tener límites de tasa)


### Nota metodológica: exploración en el portal de datos

Pasos recomendados para ubicar y validar datasets en Datos.gov.co:

1. Ir a https://datos.gov.co/
2. Buscar el conjunto por palabras clave.
3. Verificar el identificador del dataset (formato `XXXX-YYYY`).
4. Confirmar que el volumen y variables cumplan el objetivo del curso (>=5 variables numéricas y variable objetivo).

Ejemplo básico de consulta con SODA:

```python
results = client.get('2pnw-mmge', limit=1000)
df = pd.DataFrame.from_records(results)
```

En este trabajo se continúa con el dataset `2pnw-mmge` y se evalúa su preparación para aprendizaje automático.

## 3. Carga de datos

In [ ]:
def fetch_data_from_soda(domain, dataset_id, app_token=None, limit=None):
    """Obtiene datos desde Datos.gov.co usando la API SODA."""
    try:
        print(f'Conectando a {domain}...')
        client = Socrata(domain, app_token, timeout=60)

        print(f'Consultando dataset {dataset_id}...')
        if limit:
            print(f'  Límite aplicado para prueba: {limit:,} registros')
            results = client.get(dataset_id, limit=limit)
        else:
            results = client.get(dataset_id)

        if not results:
            print('Error: la consulta retornó resultados vacíos.')
            return None

        df = pd.DataFrame.from_records(results)
        print('Datos cargados correctamente.')
        print(f'  Tamaño: {df.shape[0]:,} filas x {df.shape[1]} columnas')
        return df

    except Exception as e:
        print(f'Error al obtener datos: {str(e)}')
        print('\nSugerencias de validación:')
        print(f'  1. Verifica DATASET_ID (actual: {dataset_id})')
        print('  2. Verifica conexión a internet')
        print('  3. Revisa disponibilidad del portal datos.gov.co')
        print('  4. Si usas token, valida que sea correcto')
        return None

In [ ]:
# Carga principal del dataset
print('Iniciando consulta de datos...\n')

df = fetch_data_from_soda(
    domain=DOMAIN,
    dataset_id=DATASET_ID,
    app_token=APP_TOKEN,
    limit=None  # Cambia a un valor (por ejemplo 10000) para pruebas rápidas
)

if df is not None and len(df) > 0:
    print('\nResumen inicial del dataset:')
    print(f'  Total de registros: {df.shape[0]:,}')
    print(f'  Total de columnas: {df.shape[1]}')
    print('\nPrimeras filas:')
    display(df.head())
else:
    print('\nNo se logró cargar la data desde la API.')
    print('Alternativa: cargar un CSV local con la misma estructura.')
    print('Ejemplo: df = pd.read_csv("ruta/a/archivo.csv")')

## 1. Descripción del problema y definición de variable objetivo

Esta sección define qué representa el dataset y cuál será la variable objetivo para modelado supervisado.

Criterio:
- Se prioriza una variable con significado de negocio y suficiente cobertura de datos.
- Si no se configura manualmente, se intentará inferir una candidata de forma automática.
- Se verificará si el problema corresponde a clasificación o regresión.

In [ ]:
# Configuración de la variable objetivo
TARGET_COLUMN = None  # Ejemplo: 'nombre_columna_objetivo'

if df is not None and len(df) > 0:
    candidate_names = [
        'target', 'objetivo', 'label', 'class', 'clase',
        'y', 'eva', 'riesgo', 'categoria', 'resultado'
    ]
    lower_map = {c.lower(): c for c in df.columns}

    if TARGET_COLUMN is None:
        for name in candidate_names:
            if name in lower_map:
                TARGET_COLUMN = lower_map[name]
                break

    if TARGET_COLUMN is None:
        # Heurística: variable categórica con cardinalidad moderada
        cat_candidates = [
            c for c in df.columns
            if df[c].dtype == 'object' and 2 <= df[c].nunique(dropna=True) <= 20
        ]
        if cat_candidates:
            TARGET_COLUMN = cat_candidates[0]

    print('Descripción general del dataset:')
    print(f'- Filas: {df.shape[0]:,}')
    print(f'- Columnas: {df.shape[1]}')
    print('- Cada fila representa un registro observado en la fuente oficial.')
    print('- Cada columna representa una característica/atributo del registro.')

    if TARGET_COLUMN and TARGET_COLUMN in df.columns:
        n_unique = df[TARGET_COLUMN].nunique(dropna=True)
        problem_type = 'clasificación' if (df[TARGET_COLUMN].dtype == 'object' or n_unique <= 20) else 'regresión'
        print(f"\nVariable objetivo seleccionada: {TARGET_COLUMN}")
        print(f"Tipo de problema inferido: {problem_type}")
    else:
        print('\nNo se pudo inferir automáticamente la variable objetivo.')
        print('Define manualmente TARGET_COLUMN en esta celda y vuelve a ejecutar.')
else:
    print('No hay datos cargados para describir el problema.')

In [ ]:
def load_from_csv(filepath):
    """Carga datos desde un archivo CSV local."""
    try:
        print(f'Cargando datos desde {filepath}...')
        df_local = pd.read_csv(filepath)
        print('Datos cargados correctamente.')
        print(f'  Tamaño: {df_local.shape[0]:,} filas x {df_local.shape[1]} columnas')
        return df_local
    except Exception as e:
        print(f'Error al cargar CSV: {str(e)}')
        return None

# Descomenta esta línea si quieres usar un archivo local:
# df = load_from_csv('ruta/a/tu/archivo.csv')

## 4. Inspección del dataset

In [ ]:
# Inspección de estructura y memoria
if df is not None and len(df) > 0:
    print('Información general del dataset:')
    print('=' * 80)
    df.info()
    print('\n' + '=' * 80)
    memory_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f'\nUso de memoria: {memory_mb:.2f} MB')
else:
    print('No hay datos cargados. Ejecuta primero la sección de carga.')

In [ ]:
# Resumen de tipos de datos y columnas
if df is not None and len(df) > 0:
    print('Resumen de tipos de variables:')
    print(df.dtypes.value_counts())
    print(f'\nTotal de columnas: {len(df.columns)}')
    print(f'Total de filas: {len(df):,}')
    print('\nListado de columnas:')
    for i, col in enumerate(df.columns, 1):
        print(f'  {i}. {col}')
else:
    print('No hay datos cargados.')

## 5. Calidad de los datos

En esta sección se evalúan los principales problemas de calidad. Cada hallazgo se acompaña de una justificación de corrección para dejar el dataset listo para modelado.

In [ ]:
# Evaluación integral de calidad de datos: faltantes, duplicados, imposibles e inconsistencias
if df is not None and len(df) > 0:
    print('Resumen de calidad de datos')
    print('=' * 90)

    # 1) Faltantes
    missing_data = pd.DataFrame({
        'columna': df.columns,
        'faltantes': df.isnull().sum().values,
        'porcentaje_faltante': (df.isnull().sum() / len(df) * 100).round(2).values
    }).sort_values('porcentaje_faltante', ascending=False)
    print('\n1) Valores faltantes (top 15):')
    display(missing_data.head(15))

    # 2) Duplicados exactos
    duplicate_count = int(df.duplicated().sum())
    duplicate_pct = round((duplicate_count / len(df)) * 100, 2)
    print(f'\n2) Duplicados exactos: {duplicate_count:,} ({duplicate_pct}%)')

    # 3) Valores imposibles (regla general): negativos en columnas numéricas con nombre de conteo/edad/monto
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    impossible_checks = []
    keywords = ['edad', 'age', 'cantidad', 'count', 'total', 'monto', 'valor', 'precio', 'score']

    for col in numeric_cols:
        col_lower = col.lower()
        if any(k in col_lower for k in keywords):
            impossible_n = int((df[col] < 0).sum())
            if impossible_n > 0:
                impossible_checks.append({
                    'columna': col,
                    'regla': 'valor < 0',
                    'casos': impossible_n,
                    'porcentaje': round((impossible_n / len(df)) * 100, 2)
                })

    print('\n3) Valores imposibles detectados (según reglas generales):')
    if impossible_checks:
        display(pd.DataFrame(impossible_checks).sort_values('porcentaje', ascending=False))
    else:
        print('No se detectaron imposibles con las reglas configuradas.')

    # 4) Inconsistencias básicas en texto
    object_cols = df.select_dtypes(include='object').columns.tolist()
    inconsistency_report = []

    for col in object_cols[:20]:
        s = df[col].dropna().astype(str)
        if len(s) == 0:
            continue
        spaces_issue = int((s != s.str.strip()).sum())
        case_variation = int(s.str.lower().nunique() != s.nunique())
        if spaces_issue > 0 or case_variation > 0:
            inconsistency_report.append({
                'columna': col,
                'espacios_extremos': spaces_issue,
                'variaciones_mayusculas': case_variation
            })

    print('\n4) Inconsistencias de formato en variables categóricas:')
    if inconsistency_report:
        display(pd.DataFrame(inconsistency_report))
    else:
        print('No se observaron inconsistencias evidentes de formato en las primeras variables categóricas evaluadas.')

    print('\nJustificación de correcciones sugeridas:')
    print('- Faltantes: imputar o eliminar según porcentaje y relevancia de la variable.')
    print('- Duplicados: eliminar para evitar sesgo y sobre-representación.')
    print('- Imposibles: corregir con reglas de negocio o marcar como faltantes.')
    print('- Inconsistencias: normalizar texto (strip, lower/title) para consolidar categorías.')
else:
    print('No hay datos cargados.')

In [ ]:
# Visualización del porcentaje de faltantes por columna
if df is not None and len(df) > 0:
    missing_data_plot = pd.DataFrame({
        'columna': df.columns,
        'porcentaje_faltante': (df.isnull().sum() / len(df) * 100).round(2)
    })
    missing_data_plot = missing_data_plot[missing_data_plot['porcentaje_faltante'] > 0].sort_values('porcentaje_faltante')

    if len(missing_data_plot) > 0:
        fig, ax = plt.subplots(figsize=(10, 6))
        missing_data_plot.plot(
            kind='barh',
            x='columna',
            y='porcentaje_faltante',
            ax=ax,
            color='coral',
            legend=False
        )
        ax.set_xlabel('Porcentaje de faltantes (%)')
        ax.set_title('Distribución de valores faltantes')
        plt.tight_layout()
        plt.show()
    else:
        print('No hay faltantes para visualizar.')
else:
    print('No hay datos cargados.')

## 6. Resumen estadístico inicial

In [ ]:
# Estadísticos descriptivos para variables numéricas
if df is not None and len(df) > 0:
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if len(numerical_cols) > 0:
        print('Estadística descriptiva (variables numéricas):')
        print('=' * 100)
        display(df[numerical_cols].describe().T)
    else:
        print('No se encontraron columnas numéricas.')
else:
    print('No hay datos cargados.')

In [ ]:
# Estadísticos para variables categóricas
if df is not None and len(df) > 0:
    categorical_cols = df.select_dtypes(include='object').columns.tolist()

    if len(categorical_cols) > 0:
        print('Resumen de variables categóricas:')
        print('=' * 100)
        for col in categorical_cols:
            print(f'\n{col}:')
            print(f'  Valores únicos: {df[col].nunique()}')
            print('  Valores más frecuentes:')
            print(df[col].value_counts().head())
    else:
        print('No se encontraron columnas categóricas.')
else:
    print('No hay datos cargados.')

## 7. Distribuciones y necesidad de transformaciones

In [ ]:
if df is not None and len(df) > 0:
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if len(numerical_cols) > 0:
        n_cols = len(numerical_cols)
        n_rows = (n_cols + 2) // 3

        fig, axes = plt.subplots(n_rows, 3, figsize=(15, 4 * n_rows))
        axes = axes.flatten()

        for idx, col in enumerate(numerical_cols):
            series = df[col].dropna()
            axes[idx].hist(series, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
            skew = series.skew()
            axes[idx].set_title(f'Distribución de {col} (asimetría={skew:.2f})')
            axes[idx].set_xlabel(col)
            axes[idx].set_ylabel('Frecuencia')
            axes[idx].grid(True, alpha=0.3)

        for idx in range(len(numerical_cols), len(axes)):
            axes[idx].set_visible(False)

        plt.tight_layout()
        plt.show()

        high_skew = [c for c in numerical_cols if abs(df[c].dropna().skew()) > 1]
        if high_skew:
            print('Variables con asimetría alta (|skew| > 1), candidatas a transformación:')
            print(high_skew)
        else:
            print('No se detectó asimetría alta en variables numéricas.')
    else:
        print('No hay columnas numéricas para visualizar.')
else:
    print('No hay datos cargados.')

In [ ]:
if df is not None and len(df) > 0:
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if len(numerical_cols) > 0:
        n_cols = len(numerical_cols)
        n_rows = (n_cols + 2) // 3

        fig, axes = plt.subplots(n_rows, 3, figsize=(15, 4 * n_rows))
        axes = axes.flatten()

        for idx, col in enumerate(numerical_cols):
            axes[idx].boxplot(df[col].dropna())
            axes[idx].set_title(f'Boxplot de {col}')
            axes[idx].set_ylabel(col)
            axes[idx].grid(True, alpha=0.3)

        for idx in range(len(numerical_cols), len(axes)):
            axes[idx].set_visible(False)

        plt.tight_layout()
        plt.show()
    else:
        print('No hay columnas numéricas para visualizar.')
else:
    print('No hay datos cargados.')

## 8. Relación entre variables (correlación y redundancia)

In [ ]:
if df is not None and len(df) > 0:
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if len(numerical_cols) > 1:
        corr_matrix = df[numerical_cols].corr()

        fig, ax = plt.subplots(figsize=(12, 10))
        sns.heatmap(
            corr_matrix,
            annot=True,
            fmt='.2f',
            cmap='coolwarm',
            center=0,
            square=True,
            ax=ax,
            cbar_kws={'label': 'Correlación'}
        )
        ax.set_title('Matriz de correlación - variables numéricas', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        print('\nPares altamente correlacionados (|correlación| > 0.7):')
        print('=' * 80)
        corr_pairs = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i + 1, len(corr_matrix.columns)):
                if abs(corr_matrix.iloc[i, j]) > 0.7:
                    corr_pairs.append({
                        'Variable_1': corr_matrix.columns[i],
                        'Variable_2': corr_matrix.columns[j],
                        'Correlacion': corr_matrix.iloc[i, j]
                    })

        if corr_pairs:
            corr_df = pd.DataFrame(corr_pairs)
            display(corr_df.sort_values('Correlacion', key=lambda s: s.abs(), ascending=False))
        else:
            print('No se encontraron pares altamente correlacionados.')
    else:
        print('No hay suficientes variables numéricas para análisis de correlación.')
else:
    print('No hay datos cargados.')

## 9. Variables categóricas

In [ ]:
if df is not None and len(df) > 0:
    categorical_cols = df.select_dtypes(include='object').columns.tolist()

    if len(categorical_cols) > 0:
        n_cat_cols = len(categorical_cols)
        n_rows = (n_cat_cols + 2) // 3

        fig, axes = plt.subplots(n_rows, 3, figsize=(15, 4 * n_rows))
        axes = axes.flatten()

        for idx, col in enumerate(categorical_cols):
            top_values = df[col].value_counts().head(10)
            axes[idx].barh(range(len(top_values)), top_values.values, color='lightgreen')
            axes[idx].set_yticks(range(len(top_values)))
            axes[idx].set_yticklabels(top_values.index)
            axes[idx].set_title(f'Top 10 categorías en {col}')
            axes[idx].set_xlabel('Conteo')
            axes[idx].grid(True, alpha=0.3, axis='x')

        for idx in range(len(categorical_cols), len(axes)):
            axes[idx].set_visible(False)

        plt.tight_layout()
        plt.show()
    else:
        print('No se encontraron columnas categóricas.')
else:
    print('No hay datos cargados.')

## 10. Detección de outliers

In [ ]:
if df is not None and len(df) > 0:
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if len(numerical_cols) > 0:
        outlier_summary = []

        for col in numerical_cols:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1

            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

            if len(outliers) > 0:
                outlier_summary.append({
                    'columna': col,
                    'cantidad_outliers': len(outliers),
                    'porcentaje_outliers': round((len(outliers) / len(df)) * 100, 2),
                    'limite_inferior': lower_bound,
                    'limite_superior': upper_bound
                })

        if outlier_summary:
            outlier_df = pd.DataFrame(outlier_summary).sort_values('porcentaje_outliers', ascending=False)
            print('Resumen de outliers (método IQR):')
            print('=' * 100)
            display(outlier_df)
        else:
            print('No se detectaron outliers con el método IQR.')
    else:
        print('No hay variables numéricas para detección de outliers.')
else:
    print('No hay datos cargados.')

## 11. Relación con la variable objetivo

Esta sección evalúa qué variables aportan señal para predecir la variable objetivo, y cuáles parecen poco útiles.

In [ ]:
# Separabilidad y utilidad de variables respecto al objetivo
if df is not None and len(df) > 0 and TARGET_COLUMN in df.columns:
    target = TARGET_COLUMN
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != target]

    if len(numeric_cols) == 0:
        print('No hay variables numéricas para evaluar separabilidad.')
    else:
        y = df[target]
        y_non_null = y.dropna()
        n_unique = y_non_null.nunique()
        is_classification = (y.dtype == 'object') or (n_unique <= 20)

        # Ranking simple por correlación absoluta (si objetivo numérico)
        if not is_classification and pd.api.types.is_numeric_dtype(df[target]):
            corr_target = df[numeric_cols + [target]].corr(numeric_only=True)[target].drop(target).abs().sort_values(ascending=False)
            print('Variables numéricas más relacionadas con el objetivo (|correlación|):')
            display(corr_target.to_frame('corr_abs_con_objetivo').head(10))

        # Para clasificación: diferencia de medias entre clases (ANOVA simple cuando aplica)
        else:
            print('Evaluación de separabilidad para clasificación:')
            class_counts = y.value_counts(dropna=True)
            valid_classes = class_counts[class_counts >= 10].index.tolist()
            print(f'- Clases con al menos 10 registros: {len(valid_classes)}')

            separability_rows = []
            for col in numeric_cols:
                groups = [df.loc[df[target] == cls, col].dropna().values for cls in valid_classes]
                groups = [g for g in groups if len(g) >= 10]
                if len(groups) >= 2:
                    try:
                        f_stat, p_value = stats.f_oneway(*groups)
                        separability_rows.append({
                            'variable': col,
                            'f_stat': float(f_stat),
                            'p_value': float(p_value)
                        })
                    except Exception:
                        pass

            if separability_rows:
                sep_df = pd.DataFrame(separability_rows).sort_values('f_stat', ascending=False)
                print('Top variables con mayor separabilidad entre clases (ANOVA):')
                display(sep_df.head(10))
            else:
                print('No fue posible calcular separabilidad robusta con las clases actuales.')

        # Visualización de variables candidatas útiles
        top_vars = numeric_cols[:4] if len(numeric_cols) >= 4 else numeric_cols
        if top_vars:
            n = len(top_vars)
            fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
            if n == 1:
                axes = [axes]
            for i, col in enumerate(top_vars):
                sns.boxplot(data=df, x=target, y=col, ax=axes[i])
                axes[i].set_title(f'{col} vs {target}')
                axes[i].tick_params(axis='x', rotation=45)
            plt.tight_layout()
            plt.show()

        # Señal de variables posiblemente inútiles
        low_var = [c for c in numeric_cols if df[c].nunique(dropna=True) <= 1]
        if low_var:
            print('\nVariables numéricas potencialmente inútiles (varianza nula):')
            print(low_var)
        else:
            print('\nNo se detectaron variables numéricas con varianza nula.')
else:
    print('No se puede evaluar relación con objetivo: valida que TARGET_COLUMN esté definido y exista.')

## 12. Balance entre clases e implicaciones

Si el problema es de clasificación, se analiza el desbalance de clases porque impacta la selección de métricas, estrategia de partición y técnicas de remuestreo.

In [ ]:
# Análisis de balance de clases
if df is not None and len(df) > 0 and TARGET_COLUMN in df.columns:
    y = df[TARGET_COLUMN].dropna()
    is_classification = (y.dtype == 'object') or (y.nunique() <= 20)

    if is_classification:
        class_dist = y.value_counts().to_frame('conteo')
        class_dist['porcentaje'] = (class_dist['conteo'] / class_dist['conteo'].sum() * 100).round(2)
        class_dist = class_dist.reset_index().rename(columns={'index': 'clase'})
        display(class_dist)

        ratio = class_dist['conteo'].max() / class_dist['conteo'].min()
        print(f'Ratio clase mayoritaria/minoritaria: {ratio:.2f}')

        plt.figure(figsize=(10, 4))
        sns.barplot(data=class_dist, x='clase', y='conteo', color='steelblue')
        plt.title(f'Distribución de clases en {TARGET_COLUMN}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

        print('\nImplicaciones para modelado:')
        if ratio >= 3:
            print('- Existe desbalance relevante; usar partición estratificada.')
            print('- Priorizar métricas robustas: F1 macro, balanced accuracy, ROC-AUC por clase.')
            print('- Evaluar remuestreo (SMOTE/undersampling) o ponderación de clases.')
        else:
            print('- El balance es razonable para comenzar con modelos base.')
            print('- Mantener validación estratificada y monitorear métricas por clase.')
    else:
        print('La variable objetivo parece de regresión; no aplica balance de clases.')
else:
    print('No se puede analizar balance: valida que TARGET_COLUMN esté definido y exista.')

## 13. Hallazgos, decisiones de transformación y preparación para ML

### Síntesis de resultados

1. Calidad de datos
- Se cuantificaron valores faltantes por columna y su impacto porcentual.
- Se evaluaron duplicados exactos para prevenir sesgos por sobre-representación.
- Se buscaron valores imposibles con reglas generales (por ejemplo, negativos en variables tipo conteo/edad/monto).
- Se revisaron inconsistencias de formato en texto (espacios y variaciones de mayúsculas/minúsculas).

2. Distribuciones y transformaciones
- Se analizaron histogramas y boxplots para identificar asimetría, colas largas y outliers.
- Cuando una variable presenta alta asimetría, se recomienda transformación (por ejemplo, log1p) según su dominio.
- Las decisiones de transformación deben conservar interpretabilidad y estabilidad del modelo.

3. Relación entre variables
- Se revisó la matriz de correlación para detectar redundancia y posible multicolinealidad.
- Variables altamente correlacionadas pueden eliminarse o combinarse, según impacto en desempeño y explicabilidad.

4. Relación con la variable objetivo
- Se evaluó separabilidad para identificar variables con señal predictiva.
- También se identificaron variables potencialmente inútiles (por ejemplo, varianza nula).

5. Balance de clases e implicaciones
- En clasificación, se verificó el nivel de desbalance y sus efectos en entrenamiento/evaluación.
- Si hay desbalance fuerte, se recomienda validación estratificada y métricas robustas por clase.

### Conclusión de preparación para aprendizaje automático

El dataset es **potencialmente apto para ML** si se ejecutan y documentan estas acciones antes del entrenamiento:

- [ ] Definir explícitamente la variable objetivo final (`TARGET_COLUMN`).
- [ ] Aplicar limpieza de faltantes, duplicados e inconsistencias según criterios de negocio.
- [ ] Resolver o etiquetar valores imposibles detectados.
- [ ] Decidir transformaciones para variables con asimetría/colas largas.
- [ ] Tratar redundancia o multicolinealidad cuando sea relevante.
- [ ] Confirmar estrategia frente a desbalance de clases (si aplica).

---

## Información del notebook

- Materia: Programación Avanzada
- Profesor: Jaime Alberto Vergara Tejada
- Autor: Diego Alejandro Ríos Vásquez
- Fecha de actualización: 2026-04-24
- Dataset: Colombia EVA
- Fuente: Datos.gov.co (API Socrata SODA)
- Dataset ID: 2pnw-mmge
- Propósito: EDA y alistamiento del dataset para aprendizaje automático